# 05 - Testes com LLMs

## Objetivo
Avaliar o desempenho de LLMs multimodais (OpenAI GPT-4o-mini, Claude 3.5 Sonnet 
e DeepSeek) na classificação do nível psicogenético da escrita infantil.

## Entrada
- `data/metadata/dataset.csv` — com labels reais anotados
- `data/raw/` — imagens no padrão `Cxxx_T001_IMGxxx.jpg`

## Saída
- `data/llm_results/openai/resultados.csv`
- `data/llm_results/claude/resultados.csv`
- `data/llm_results/deepseek/resultados.csv`
- `data/llm_results/comparativo_llms.csv`

In [4]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

#  ADICIONAR: apontar para os pacotes instalados no F:
sys.path.insert(0, r"F:\pacotes_python")

# Ajustar caminho para importar src/llms
sys.path.append(os.path.abspath(".."))

# Carregar variáveis de ambiente (.env na raiz)
load_dotenv("../.env")

# Imports dos módulos do projeto
from src.llms.avaliar_llms import avaliar_llm, calcular_metricas
from src.llms.openai_client import classificar_com_openai
from src.llms.claude_client import classificar_com_claude

print(" Imports OK")
print("OPENAI_API_KEY:", "OK" if os.getenv("OPENAI_API_KEY") else " FALTANDO")
print("ANTHROPIC_API_KEY:", "OK" if os.getenv("ANTHROPIC_API_KEY") else " FALTANDO")
print("DEEPSEEK_API_KEY:", "OK" if os.getenv("DEEPSEEK_API_KEY") else " FALTANDO")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
 Imports OK
OPENAI_API_KEY: OK
ANTHROPIC_API_KEY: OK
DEEPSEEK_API_KEY: OK


In [5]:
df = pd.read_csv("../data/metadata/dataset.csv")
print(f"Total de imagens: {len(df)}")
print(df["label"].value_counts())

Total de imagens: 17
label
SILABICO               11
PRE_SILABICO            3
SILABICO_ALFABETICO     3
Name: count, dtype: int64


In [6]:
caminho_teste = os.path.join("../data/raw", df.iloc[0]["filename"])
print(f"Testando: {caminho_teste}")
print(f"Existe? {os.path.exists(caminho_teste)}\n")
resultado = classificar_com_openai(caminho_teste)
print(resultado)

Testando: ../data/raw\C001_T001_IMG001.jpg
Existe? True



APIConnectionError: Connection error.

In [7]:
import urllib.request
try:
    r = urllib.request.urlopen("https://api.openai.com/v1/models", timeout=10)
    print("Status:", r.status)
except Exception as e:
    print("Erro:", type(e).__name__, e)

Erro: HTTPError HTTP Error 401: Unauthorized


In [8]:
import openai
print("Versão do openai:", openai.__version__)
print("Local:", openai.__file__)

Versão do openai: 3.13.0
Local: F:\pacotes_python\openai\__init__.py


In [1]:
import sys
sys.path.insert(0, r"F:\pacotes_python")

import openai
print("Versão:", openai.__version__)

Versão: 1.54.4


In [1]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# Pacotes instalados no F:
sys.path.insert(0, r"F:\pacotes_python")

# Módulos do projeto (src/)
sys.path.append(os.path.abspath(".."))

# Carregar variáveis de ambiente do .env na raiz
load_dotenv("../.env")

# Imports dos módulos do projeto
from src.llms.avaliar_llms import avaliar_llm, calcular_metricas
from src.llms.openai_client import classificar_com_openai
from src.llms.claude_client import classificar_com_claude

# Verificação final
import openai
print("=" * 50)
print("VERIFICAÇÃO DE AMBIENTE")
print("=" * 50)
print(f"✅ openai versão: {openai.__version__}")
print(f"✅ Local do openai: {openai.__file__}")
print(f"✅ Imports OK")
print(f"OPENAI_API_KEY:      {'OK' if os.getenv('OPENAI_API_KEY') else '❌ FALTANDO'}")
print(f"ANTHROPIC_API_KEY:   {'OK' if os.getenv('ANTHROPIC_API_KEY') else '❌ FALTANDO'}")
print("=" * 50)

VERIFICAÇÃO DE AMBIENTE
✅ openai versão: 1.60.0
✅ Local do openai: F:\pacotes_python\openai\__init__.py
✅ Imports OK
OPENAI_API_KEY:      OK
ANTHROPIC_API_KEY:   OK


In [2]:


# 1. Carregar dataset
df = pd.read_csv("../data/metadata/dataset.csv")
print(f"Total de imagens: {len(df)}")
print(df["label"].value_counts())
print("=" * 60)

# 2. Teste rápido com 1 imagem (validação)
caminho_teste = os.path.join("../data/raw", df.iloc[0]["filename"])
print(f"\nTestando 1 imagem: {caminho_teste}")
print(f"Existe? {os.path.exists(caminho_teste)}")
try:
    teste = classificar_com_openai(caminho_teste)
    print(f"Resposta OpenAI: {teste}\n")
except Exception as e:
    print(f"❌ Erro no teste: {type(e).__name__}: {e}")
    raise

# 3. Rodar OpenAI em TODAS as imagens
print("=" * 60)
print("RODANDO OPENAI EM TODAS AS IMAGENS...")
print("=" * 60)
os.makedirs("../data/llm_results/openai", exist_ok=True)

res_openai = avaliar_llm(
    df=df,
    pasta_imagens="../data/raw",
    funcao_llm=classificar_com_openai,
    nome_llm="openai-gpt-4o-mini"
)
res_openai.to_csv("../data/llm_results/openai/resultados.csv", index=False)

metricas_openai = calcular_metricas(res_openai)
print(f"\n OPENAI:")
print(f"  Acurácia: {metricas_openai['acuracia']:.3f}")
print(f"  F1-macro: {metricas_openai['f1_macro']:.3f}")
print(f"  Matriz de confusão:")
for linha in metricas_openai["matriz_confusao"]:
    print(f"    {linha}")

# 4. Rodar Claude em TODAS as imagens
print("\n" + "=" * 60)
print("RODANDO CLAUDE EM TODAS AS IMAGENS...")
print("=" * 60)
os.makedirs("../data/llm_results/claude", exist_ok=True)

res_claude = avaliar_llm(
    df=df,
    pasta_imagens="../data/raw",
    funcao_llm=classificar_com_claude,
    nome_llm="claude-3-5-sonnet"
)
res_claude.to_csv("../data/llm_results/claude/resultados.csv", index=False)

metricas_claude = calcular_metricas(res_claude)
print(f"\n CLAUDE:")
print(f"  Acurácia: {metricas_claude['acuracia']:.3f}")
print(f"  F1-macro: {metricas_claude['f1_macro']:.3f}")
print(f"  Matriz de confusão:")
for linha in metricas_claude["matriz_confusao"]:
    print(f"    {linha}")

# 5. Tabela comparativa final
print("\n" + "=" * 60)
print("COMPARATIVO FINAL DOS LLMs")
print("=" * 60)

todos = pd.concat([res_openai, res_claude], ignore_index=True)
comparativo = todos.groupby("llm").apply(
    lambda x: pd.Series(calcular_metricas(x))
).reset_index()
comparativo.columns = ["LLM", "Acurácia", "F1-macro", "Matriz"]
comparativo["Acurácia"] = comparativo["Acurácia"].round(3)
comparativo["F1-macro"] = comparativo["F1-macro"].round(3)

comparativo.to_csv("../data/llm_results/comparativo_llms.csv", index=False)
print(comparativo[["LLM", "Acurácia", "F1-macro"]].to_string(index=False))
print("=" * 60)
print("✅ Experimento concluído!")

Total de imagens: 17
label
SILABICO               11
PRE_SILABICO            3
SILABICO_ALFABETICO     3
Name: count, dtype: int64

Testando 1 imagem: ../data/raw\C001_T001_IMG001.jpg
Existe? True
Resposta OpenAI: {'classificacao': 'SILABICO_ALFABETICO', 'justificativa': 'A criança demonstra uma compreensão de que as sílabas podem ser representadas por letras, mas ainda oscila entre a escrita silábica e alfabética, apresentando algumas palavras legíveis e outras com erros. A presença de tentativas de dividir sílabas em fonemas é evidente.', 'confianca': 0.85}

RODANDO OPENAI EM TODAS AS IMAGENS...


openai-gpt-4o-mini: 100%|██████████| 17/17 [00:51<00:00,  3.00s/it]



 OPENAI:
  Acurácia: 0.118
  F1-macro: 0.083
  Matriz de confusão:
    [0, 0, 3, 0]
    [0, 1, 5, 0]
    [0, 0, 1, 0]
    [0, 0, 0, 0]

RODANDO CLAUDE EM TODAS AS IMAGENS...


claude-3-5-sonnet: 100%|██████████| 17/17 [00:16<00:00,  1.03it/s]


 CLAUDE:
  Acurácia: 0.000
  F1-macro: 0.000
  Matriz de confusão:
    [0, 0, 0, 0]
    [0, 0, 0, 0]
    [0, 0, 0, 0]
    [0, 0, 0, 0]

COMPARATIVO FINAL DOS LLMs
               LLM  Acurácia  F1-macro
 claude-3-5-sonnet     0.000     0.000
openai-gpt-4o-mini     0.118     0.083
✅ Experimento concluído!



C:\Users\thela\AppData\Local\Temp\ipykernel_14476\679699053.py:68: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comparativo = todos.groupby("llm").apply(


In [3]:
print("=== AMOSTRA DOS RESULTADOS DO CLAUDE ===")
print(res_claude[["image_id", "label_real", "label_predito", "confianca"]].head(20).to_string())
print("\n=== VALORES ÚNICOS DE label_predito ===")
print(res_claude["label_predito"].value_counts())

=== AMOSTRA DOS RESULTADOS DO CLAUDE ===
   image_id           label_real label_predito  confianca
0    IMG001         PRE_SILABICO          ERRO        0.0
1    IMG002         PRE_SILABICO          ERRO        0.0
2    IMG003             SILABICO          ERRO        0.0
3    IMG004             SILABICO          ERRO        0.0
4    IMG005             SILABICO          ERRO        0.0
5    IMG006         PRE_SILABICO          ERRO        0.0
6    IMG007             SILABICO          ERRO        0.0
7    IMG008             SILABICO          ERRO        0.0
8    IMG009             SILABICO          ERRO        0.0
9    IMG010             SILABICO          ERRO        0.0
10   IMG011             SILABICO          ERRO        0.0
11   IMG012  SILABICO_ALFABETICO          ERRO        0.0
12   IMG013             SILABICO          ERRO        0.0
13   IMG014             SILABICO          ERRO        0.0
14   IMG015             SILABICO          ERRO        0.0
15   IMG016  SILABICO_ALFABETIC

In [4]:
import unicodedata

def normalizar_label(label):
    if not isinstance(label, str):
        return label
    # Remove acentos
    label = unicodedata.normalize("NFKD", label).encode("ASCII", "ignore").decode("ASCII")
    # Padroniza separadores e caixa
    label = label.upper().replace("-", "_").replace(" ", "_")
    return label

res_claude["label_predito"] = res_claude["label_predito"].apply(normalizar_label)
metricas_claude = calcular_metricas(res_claude)
print(f"Acurácia Claude (normalizada): {metricas_claude['acuracia']:.3f}")
print(f"F1-macro Claude (normalizada): {metricas_claude['f1_macro']:.3f}")
for linha in metricas_claude["matriz_confusao"]:
    print(linha)

Acurácia Claude (normalizada): 0.000
F1-macro Claude (normalizada): 0.000
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]


In [2]:
# ============================================================
# RODAR CLAUDE EM TODAS AS IMAGENS (versão autocontida)
# ============================================================

import os
import sys
import pandas as pd
from dotenv import load_dotenv

# ⚠️ Configuração de caminhos (essencial após reiniciar o kernel)
sys.path.insert(0, r"F:\pacotes_python")   # pacotes instalados no F:
sys.path.append(os.path.abspath(".."))     # pasta src/ do projeto
load_dotenv("../.env")

# Imports do projeto
from src.llms.avaliar_llms import avaliar_llm, calcular_metricas
from src.llms.claude_client import classificar_com_claude

# Recarregar dataset
df = pd.read_csv("../data/metadata/dataset.csv")

# Criar pasta
os.makedirs("../data/llm_results/claude", exist_ok=True)

# Rodar Claude
print("=" * 60)
print("RODANDO CLAUDE EM TODAS AS IMAGENS...")
print("=" * 60)

res_claude = avaliar_llm(
    df=df,
    pasta_imagens="../data/raw",
    funcao_llm=classificar_com_claude,
    nome_llm="claude-sonnet-4-5"
)

# Salvar resultados
res_claude.to_csv("../data/llm_results/claude/resultados.csv", index=False)

# Calcular métricas
metricas_claude = calcular_metricas(res_claude)
print(f"\n📊 CLAUDE:")
print(f"  Acurácia: {metricas_claude['acuracia']:.3f}")
print(f"  F1-macro: {metricas_claude['f1_macro']:.3f}")
print(f"  Matriz de confusão:")
for linha in metricas_claude["matriz_confusao"]:
    print(f"    {linha}")

# Verificar erros
print(f"\n=== Valores únicos de label_predito ===")
print(res_claude["label_predito"].value_counts())
print("\n=== Primeiras 5 linhas ===")
print(res_claude[["image_id", "label_real", "label_predito", "confianca"]].head(5).to_string())

RODANDO CLAUDE EM TODAS AS IMAGENS...


claude-sonnet-4-5: 100%|██████████| 17/17 [01:43<00:00,  6.11s/it]


📊 CLAUDE:
  Acurácia: 0.059
  F1-macro: 0.071
  Matriz de confusão:
    [0, 1, 0, 2]
    [0, 0, 3, 8]
    [1, 0, 1, 1]
    [0, 0, 0, 0]

=== Valores únicos de label_predito ===
label_predito
ALFABETICO             11
SILABICO_ALFABETICO     4
SILABICO                1
PRE_SILABICO            1
Name: count, dtype: int64

=== Primeiras 5 linhas ===
  image_id    label_real label_predito  confianca
0   IMG001  PRE_SILABICO    ALFABETICO       0.95
1   IMG002  PRE_SILABICO    ALFABETICO       0.95
2   IMG003      SILABICO    ALFABETICO       0.95
3   IMG004      SILABICO    ALFABETICO       0.95
4   IMG005      SILABICO    ALFABETICO       0.95


In [3]:
# Consolidar resultados finais dos dois LLMs
import pandas as pd

res_openai = pd.read_csv("../data/llm_results/openai/resultados.csv")
res_claude = pd.read_csv("../data/llm_results/claude/resultados.csv")

todos = pd.concat([res_openai, res_claude], ignore_index=True)

comparativo = todos.groupby("llm").apply(
    lambda x: pd.Series({
        "Acurácia": round((x["label_real"] == x["label_predito"]).mean(), 3),
        "F1-macro": round(__import__("sklearn.metrics", fromlist=["f1_score"]).f1_score(
            x["label_real"], x["label_predito"],
            average="macro",
            labels=["PRE_SILABICO", "SILABICO", "SILABICO_ALFABETICO", "ALFABETICO"],
            zero_division=0
        ), 3)
    })
).reset_index()
comparativo.columns = ["LLM", "Acurácia", "F1-macro"]

comparativo.to_csv("../data/llm_results/comparativo_llms.csv", index=False)
print(comparativo.to_string(index=False))

               LLM  Acurácia  F1-macro
 claude-sonnet-4-5     0.059     0.071
openai-gpt-4o-mini     0.118     0.083


C:\Users\thela\AppData\Local\Temp\ipykernel_13704\211053960.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comparativo = todos.groupby("llm").apply(


In [4]:
import os
import pandas as pd
from sklearn.metrics import f1_score

# Caminhos
caminhos = {
    "OpenAI": "../data/llm_results/openai/resultados.csv",
    "Claude": "../data/llm_results/claude/resultados.csv",
    "Comparativo": "../data/llm_results/comparativo_llms.csv",
}

# Tenta ler os arquivos existentes
res_openai = None
res_claude = None

try:
    if os.path.exists(caminhos["OpenAI"]):
        res_openai = pd.read_csv(caminhos["OpenAI"])
        if res_openai.empty:
            res_openai = None
except Exception:
    res_openai = None

try:
    if os.path.exists(caminhos["Claude"]):
        res_claude = pd.read_csv(caminhos["Claude"])
        if res_claude.empty:
            res_claude = None
except Exception:
    res_claude = None

# Recriar se necessário
if res_openai is None:
    print("Rodando OpenAI novamente...")
    from src.llms.openai_client import classificar_com_openai
    from src.llms.avaliar_llms import avaliar_llm
    df = pd.read_csv("../data/metadata/dataset.csv")
    os.makedirs("../data/llm_results/openai", exist_ok=True)
    res_openai = avaliar_llm(df, "../data/raw", classificar_com_openai, "openai-gpt-4o-mini")
    res_openai.to_csv(caminhos["OpenAI"], index=False)

if res_claude is None:
    print("Rodando Claude novamente...")
    from src.llms.claude_client import classificar_com_claude
    from src.llms.avaliar_llms import avaliar_llm
    df = pd.read_csv("../data/metadata/dataset.csv")
    os.makedirs("../data/llm_results/claude", exist_ok=True)
    res_claude = avaliar_llm(df, "../data/raw", classificar_com_claude, "claude-sonnet-4-5")
    res_claude.to_csv(caminhos["Claude"], index=False)

# Consolidar comparativo
todos = pd.concat([res_openai, res_claude], ignore_index=True)

comparativo = todos.groupby("llm").apply(
    lambda x: pd.Series({
        "Acurácia": round((x["label_real"] == x["label_predito"]).mean(), 3),
        "F1-macro": round(f1_score(
            x["label_real"], x["label_predito"],
            average="macro",
            labels=["PRE_SILABICO", "SILABICO", "SILABICO_ALFABETICO", "ALFABETICO"],
            zero_division=0
        ), 3)
    })
).reset_index()
comparativo.columns = ["LLM", "Acurácia", "F1-macro"]
comparativo.to_csv(caminhos["Comparativo"], index=False)

print("Arquivos regravados com sucesso.")
print(f"OpenAI: {len(res_openai)} linhas")
print(f"Claude: {len(res_claude)} linhas")
print(f"Comparativo: {len(comparativo)} linhas")
print()
print("Tabela comparativa final:")
print(comparativo.to_string(index=False))

Arquivos regravados com sucesso.
OpenAI: 17 linhas
Claude: 17 linhas
Comparativo: 2 linhas

Tabela comparativa final:
               LLM  Acurácia  F1-macro
 claude-sonnet-4-5     0.059     0.071
openai-gpt-4o-mini     0.118     0.083


C:\Users\thela\AppData\Local\Temp\ipykernel_13704\1487207531.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comparativo = todos.groupby("llm").apply(
